In [1]:
import subprocess, requests, time, os

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "deepseek-coder:6.7b"
MAX_ATTEMPTS = 5
LEAN_FILE = "Attempt.lean"
PARTIAL_FILE = "Partial.lean"
PROJECT_DIR = os.path.expanduser("~/llm-local/lean_gcd_baby")

SYSTEM_PROMPT = """You are an expert in Lean 4 and Mathlib.
Output ONLY raw Lean 4 tactic lines. 
NEVER use markdown. NEVER write ``` or ```lean.
NEVER repeat the theorem statement or import line.
Just the raw tactic lines, one per line, nothing else."""

print("Config loaded.")
print(f"Project dir: {PROJECT_DIR}")
print(f"Model: {MODEL}")

Config loaded.
Project dir: /Users/sashivbhatia/llm-local/lean_gcd_baby
Model: deepseek-coder:6.7b


In [2]:
THEOREMS = {
    "warmup": {
        "description": "Prove that n + 0 = n for all natural numbers",
        "header": "import Mathlib\ntheorem add_zero_right (n : ℕ) : n + 0 = n := by\n",
    },
    "easy": {
        "description": "Prove that n * 0 = 0 for all natural numbers",
        "header": "import Mathlib\ntheorem mul_zero_right (n : ℕ) : n * 0 = 0 := by\n",
    },
    "medium": {
        "description": "Prove that n + m = m + n for all natural numbers",
        "header": "import Mathlib\ntheorem add_comm_nat (n m : ℕ) : n + m = m + n := by\n",
    },
    "stretch": {
        "description": "Prove that gcd(n,m) divides both n and m",
        "header": "import Mathlib\ntheorem gcd_dvd (n m : ℕ) : Nat.gcd n m ∣ n ∧ Nat.gcd n m ∣ m := by\n",
    },
}

print("Theorems loaded:")
for name, t in THEOREMS.items():
    print(f"  {name}: {t['description']}")

Theorems loaded:
  warmup: Prove that n + 0 = n for all natural numbers
  easy: Prove that n * 0 = 0 for all natural numbers
  medium: Prove that n + m = m + n for all natural numbers
  stretch: Prove that gcd(n,m) divides both n and m


In [3]:
def run_lean(filepath, timeout=60):
    """Run lean on a file. Returns (returncode, output)."""
    try:
        result = subprocess.run(
            ["lake", "env", "lean", filepath],
            capture_output=True, text=True,
            timeout=timeout, cwd=PROJECT_DIR
        )
        output = (result.stdout + result.stderr).strip()
        return result.returncode, output
    except subprocess.TimeoutExpired:
        return -1, "timeout"

def check_partial(header, tactics_so_far):
    """
    Append sorry and check if what we have so far is valid.
    Returns (ok, error_message).
    """
    code = header + tactics_so_far + "  sorry\n"
    filepath = os.path.join(PROJECT_DIR, PARTIAL_FILE)
    with open(filepath, "w") as f:
        f.write(code)
    returncode, output = run_lean(filepath, timeout=30)
    has_error = (
        "error:" in output.lower() and
        "declaration uses 'sorry'" not in output
    )
    if has_error:
        return False, output
    return True, ""

def check_complete(header, tactics):
    """Check the full proof with no sorry."""
    code = header + tactics
    filepath = os.path.join(PROJECT_DIR, LEAN_FILE)
    with open(filepath, "w") as f:
        f.write(code)
    returncode, output = run_lean(filepath, timeout=60)
    success = (
        returncode == 0 and
        "sorry" not in tactics and
        "error:" not in output.lower()
    )
    return success, output

# Quick sanity check — write and verify a trivial proof
test_path = os.path.join(PROJECT_DIR, "Test.lean")
with open(test_path, "w") as f:
    f.write("import Mathlib\ntheorem test (n : ℕ) : n + 0 = n := by simp\n")
rc, out = run_lean(test_path)
if rc == 0:
    print("Lean checker working!")
else:
    print(f"Something wrong with Lean: {out}")

Lean checker working!


In [4]:
def clean_token_stream(text):
    """Remove markdown code fences that the model insists on adding."""
    text = text.replace("```lean", "").replace("```", "")
    return text

def stream_and_check(prompt, header):
    tactics_so_far = ""
    current_line = ""
    line_count = 0

    print("Streaming proof (tactic-level checking active)...")
    print("─" * 50)

    with requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "prompt": prompt,
            "system": SYSTEM_PROMPT,
            "stream": True,
            "options": {"temperature": 0.2, "num_predict": 400}
        },
        stream=True
    ) as response:

        for raw_line in response.iter_lines():
            if not raw_line:
                continue
            try:
                chunk = requests.compat.json.loads(raw_line)
            except Exception:
                continue

            token = chunk.get("response", "")
            done = chunk.get("done", False)

            # Strip markdown fences from each token before processing
            token = clean_token_stream(token)

            print(token, end="", flush=True)

            current_line += token
            tactics_so_far += token

            if "\n" in current_line:
                line_count += 1
                current_line = ""

                # Skip blank lines, very first line, and sorry placeholders
                stripped = tactics_so_far.strip()
                if not stripped or line_count < 2:
                    continue
                if "sorry" in tactics_so_far:
                    continue
                # Skip lines that are just import statements
                # (model sometimes echoes the header back)
                last_line = [l for l in tactics_so_far.split("\n") if l.strip()]
                if last_line and last_line[-1].strip().startswith("import"):
                    continue
                if last_line and last_line[-1].strip().startswith("theorem"):
                    continue

                ok, error = check_partial(header, tactics_so_far)
                if not ok:
                    print(f"\n\n[!] Bad tactic caught at line {line_count} — aborting early")
                    print(f"Lean says: {error[:200]}")
                    return tactics_so_far, error

            if done:
                break

    print()
    return tactics_so_far, None

print("Updated stream_and_check ready.")

Updated stream_and_check ready.


In [5]:
def build_prompt(theorem, error_history):
    desc = theorem["description"]

    if not error_history:
        return f"""Prove this theorem in Lean 4 using Mathlib.

Task: {desc}

Write ONLY the tactic lines inside the `by` block.
Do NOT repeat the theorem statement or `import Mathlib`.
One tactic per line. Useful tactics:
- `simp` for simplification
- `omega` for arithmetic on natural numbers
- `ring` for algebraic identities
- `constructor` then prove each part for ∧ goals
- `exact Nat.add_comm n m` for commutativity
- `exact ⟨Nat.gcd_dvd_left n m, Nat.gcd_dvd_right n m⟩` for gcd

Tactics only, starting now:"""

    else:
        errors_fmt = "\n\n".join([
            f"Attempt {i+1} error:\n{e[:300]}"
            for i, e in enumerate(error_history)
        ])
        return f"""Prove this Lean 4 theorem: {desc}

Previous attempts failed:
{errors_fmt}

Try these fixes:
- For arithmetic: use `omega`
- For `n + 0 = n`: try `simp` or `rfl`
- For ∧ goals: use `constructor` then solve each part
- For gcd: `exact ⟨Nat.gcd_dvd_left n m, Nat.gcd_dvd_right n m⟩`
- For commutativity: `exact Nat.add_comm n m`

Write ONLY tactic lines. Starting now:"""

print("Prompt builder ready.")

Prompt builder ready.


In [6]:
def prove(theorem_name):
    theorem = THEOREMS[theorem_name]
    header = theorem["header"]

    print(f"\n{'='*56}")
    print(f"Target: {theorem['description']}")
    print(f"{'='*56}")

    error_history = []

    for attempt in range(1, MAX_ATTEMPTS + 1):
        print(f"\nAttempt {attempt}/{MAX_ATTEMPTS}")
        print("─" * 40)

        prompt = build_prompt(theorem, error_history)
        tactics, abort_reason = stream_and_check(prompt, header)

        if abort_reason:
            print(f"\nAborted early. Will retry with error context...")
            error_history.append(f"Aborted at:\n{tactics}\nReason:\n{abort_reason}")
            time.sleep(1)
            continue

        print(f"\nChecking complete proof...")
        success, output = check_complete(header, tactics)

        if success:
            print(f"\n✓ PROOF SUCCEEDED on attempt {attempt}!")
            print(f"\nFinal proof:")
            print(f"{header}{tactics}")
            return True
        else:
            print(f"\n✗ Proof complete but Lean rejected it.")
            print(f"Lean says: {output[:300]}")
            error_history.append(output[:300])
            time.sleep(1)

    print(f"\nFailed after {MAX_ATTEMPTS} attempts.")
    return False

print("Prove function ready.")

Prove function ready.


In [7]:
# Run just the warmup theorem
result = prove("warmup")
print(f"\nResult: {'PASS' if result else 'FAIL'}")


Target: Prove that n + 0 = n for all natural numbers

Attempt 1/5
────────────────────────────────────────
Streaming proof (tactic-level checking active)...
──────────────────────────────────────────────────
1. simp [add_zero]
2. omega


[!] Bad tactic caught at line 2 — aborting early
Lean says: /Users/sashivbhatia/llm-local/lean_gcd_baby/Partial.lean:2:48: error: unexpected token; expected '{' or tactic
/Users/sashivbhatia/llm-local/lean_gcd_baby/Partial.lean:2:46: error: unsolved goals
n : 

Aborted early. Will retry with error context...

Attempt 2/5
────────────────────────────────────────
Streaming proof (tactic-level checking active)...
──────────────────────────────────────────────────
n  : ℕ
⊢ n + 0 = n


[!] Bad tactic caught at line 2 — aborting early
Lean says: /Users/sashivbhatia/llm-local/lean_gcd_baby/Partial.lean:3:1: error: unknown tactic
/Users/sashivbhatia/llm-local/lean_gcd_baby/Partial.lean:2:46: error: unsolved goals
n : ℕ
⊢ n + 0 = n

Aborted early. Will retry w

In [8]:
SYSTEM_PROMPT = """You are a Lean 4 tactic writer.
Your ONLY job is to write tactic lines that go inside a `by` block.

RULES:
- Output ONLY tactic names like: simp, omega, ring, rfl, constructor, exact ...
- Do NOT number your lines (no "1." or "2.")
- Do NOT output goal states (no "n : ℕ" or "⊢ ...")
- Do NOT output import statements
- Do NOT output theorem statements
- Do NOT use markdown or backticks
- One tactic per line, nothing else

EXAMPLE of correct output for `n + 0 = n`:
simp

EXAMPLE of correct output for `n + m = m + n`:
exact Nat.add_comm n m

EXAMPLE of correct output for a conjunction goal:
constructor
exact Nat.gcd_dvd_left n m
exact Nat.gcd_dvd_right n m"""


def is_garbage_line(line):
    """Filter out lines the model writes that aren't tactics."""
    s = line.strip()
    if not s:
        return True
    # Numbered list items
    if s[0].isdigit() and '.' in s[:3]:
        return True
    # Goal state lines (contain ⊢ or look like type declarations)
    if '⊢' in s:
        return True
    if ' : ' in s and not s.startswith('exact') and not s.startswith('have'):
        return True
    # Import or theorem lines
    if s.startswith('import') or s.startswith('theorem') or s.startswith('lemma'):
        return True
    # Markdown
    if s.startswith('```') or s.startswith('#'):
        return True
    return False


def stream_and_check(prompt, header):
    tactics_so_far = ""
    current_line = ""
    line_count = 0

    print("Streaming proof (tactic-level checking active)...")
    print("─" * 50)

    with requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "prompt": prompt,
            "system": SYSTEM_PROMPT,
            "stream": True,
            "options": {"temperature": 0.2, "num_predict": 400}
        },
        stream=True
    ) as response:

        for raw_line in response.iter_lines():
            if not raw_line:
                continue
            try:
                chunk = requests.compat.json.loads(raw_line)
            except Exception:
                continue

            token = chunk.get("response", "")
            done = chunk.get("done", False)

            # Clean markdown fences
            token = token.replace("```lean", "").replace("```", "")

            print(token, end="", flush=True)
            current_line += token
            tactics_so_far += token

            if "\n" in current_line:
                line_count += 1
                last_line = current_line.split("\n")[0]
                current_line = ""

                # Skip lines that are clearly not tactics
                if is_garbage_line(last_line):
                    # Remove the garbage line from tactics_so_far
                    tactics_so_far = "\n".join([
                        l for l in tactics_so_far.split("\n")
                        if not is_garbage_line(l)
                    ]) + "\n"
                    continue

                if not tactics_so_far.strip() or line_count < 2:
                    continue
                if "sorry" in tactics_so_far:
                    continue

                ok, error = check_partial(header, tactics_so_far)
                if not ok:
                    print(f"\n\n[!] Bad tactic caught at line {line_count} — aborting early")
                    print(f"Lean says: {error[:200]}")
                    return tactics_so_far, error

            if done:
                break

    print()
    return tactics_so_far, None

print("Fixed stream_and_check ready.")

Fixed stream_and_check ready.


In [ ]:
result = prove("warmup")
print(f"\nResult: {'PASS' if result else 'FAIL'}")


Target: Prove that n + 0 = n for all natural numbers

Attempt 1/5
────────────────────────────────────────
Streaming proof (tactic-level checking active)...
──────────────────────────────────────────────────
simp
ω


[!] Bad tactic caught at line 2 — aborting early
Lean says: /Users/sashivbhatia/llm-local/lean_gcd_baby/Partial.lean:4:1: error: unknown tactic

Aborted early. Will retry with error context...

Attempt 2/5
────────────────────────────────────────
Streaming proof (tactic-level checking active)...
──────────────────────────────────────────────────
simp
ω


[!] Bad tactic caught at line 2 — aborting early
Lean says: /Users/sashivbhatia/llm-local/lean_gcd_baby/Partial.lean:4:1: error: unknown tactic

Aborted early. Will retry with error context...
